# Person 2: sections 3, 4, 5 and 8

Paste these cells into the main notebook in order (3, 4, 5, then 8 at the end).
The first code cell is the setup these sections need; merge its imports into the main setup section.
All settings (lags, windows, folds, learning rates, shards, workers) are read from `config.py`.

**Task:** forecast **how many taxi pickups happen in each zone in each hour** (Part 1 produced this table).
**Splits (by time, never random):** development = January to March, test = April (scored once at the end).
Settings are chosen on **rolling validation folds** inside the development months.

In [ ]:
import config                      # first: it pins numpy to one thread per process
import time
import numpy as np
import pandas as pd

from src.features import (FEATURES, build_features, month_rows, fold_boundaries, to_arrays,
                          fit_scaler, apply_scaler)
from src.model import (make_shards, train_sequential, train_parallel, shard_gradient, scores,
                       error_scores, make_grid, search_sequential, search_parallel, best_trial,
                       load_imbalance)

demand = pd.read_parquet(config.RESULTS_DIR / "demand_zone_hour.parquet")   # Part 1's output
print(f"Demand table: {len(demand):,} rows = {demand['zone'].nunique()} zones x {demand['hour'].nunique():,} hours, "
      f"{demand['trips'].sum():,} pickups")

## 3. Features

The target is `trips` (pickups in this zone in this hour). Each (zone, hour) row gets 13 features:

| Group | Features | Meaning |
|---|---|---|
| Lag | `lag_1, lag_2, lag_3, lag_24, lag_48, lag_168` | this zone's demand 1, 2, 3 hours ago, and at the same hour 1 day, 2 days, 1 week ago |
| Rolling | `roll_mean_3, roll_mean_24, roll_std_24` | mean demand over the previous 3 and 24 hours, and how much it varied over the previous 24 hours |
| Calendar | `hour_of_day, dayofweek, is_weekend, is_holiday` | known from the clock and a holiday list |

**Why there is no leakage:** a row predicts the hour it sits on, so that hour's own pickups must never be a
feature. Every lag is a `shift(k)` of the same zone's own series with `k >= 1`, and the rolling windows are
computed on a series that was shifted by 1 first, so they stop at the *previous* hour. Calendar features
come from the clock, not from the data. The first week of January has no 168-hour history, so those rows are dropped.
The check below proves it: changing the pickups inside one hour does not change that hour's features.

In [ ]:
df = build_features(demand)
print(f"Rows before features: {len(demand):,}   after: {len(df):,}   "
      f"(dropped {len(demand) - len(df):,} rows from the first week, which has no 168-hour history)")
print(df[["hour", "zone", "trips"] + FEATURES].head(3).T.to_string())

In [ ]:
# Leakage check: add 1000 pickups to every zone in one hour, rebuild, compare that hour's features.
hour = df["hour"].iloc[len(df) // 2]
changed = demand.copy()
changed.loc[changed["hour"] == hour, "trips"] += 1000
df_changed = build_features(changed)

before = df[df["hour"] == hour][FEATURES].reset_index(drop=True)
after = df_changed[df_changed["hour"] == hour][FEATURES].reset_index(drop=True)
print("Features of that hour unchanged when its own pickups change:", before.equals(after))
assert before.equals(after)

### Time-based splits and scaling

* **Development** = January to March, **test** = April. April is not touched until section 8.
* **Rolling validation folds** (from `config.py`): each fold trains on *everything before* its start date and is scored on the
  14 days after it. The folds start on Feb 1, Feb 15, Mar 1 and Mar 15, so the model is always asked to forecast
  the future from the past, never the other way round. A setting's score is its average over the four folds.
* **Scaling:** features are standardised (subtract mean, divide by standard deviation). The mean and standard
  deviation come from the **training rows only** (inside each fold, and from January to March for the final model),
  so later months never leak into training. Scaled features also keep gradient descent stable.

In [ ]:
dev = month_rows(df, config.DEV_MONTHS)
test = month_rows(df, config.TEST_MONTHS)
X_dev_raw, y_dev, zones_dev = to_arrays(dev)
X_test_raw, y_test, _ = to_arrays(test)

folds = fold_boundaries(dev["hour"].to_numpy(), config.VAL_FOLD_STARTS, config.VAL_FOLD_DAYS)
fold_table = pd.DataFrame({
    "fold start": config.VAL_FOLD_STARTS,
    "train rows": [t for t, _ in folds],
    "validation rows": [v - t for t, v in folds],
})
print(fold_table.to_string(index=False))
print(f"\nDevelopment rows (Jan-Mar): {len(y_dev):,}   Test rows (Apr): {len(y_test):,}")

mean, std = fit_scaler(X_dev_raw)
X_dev = apply_scaler(X_dev_raw, mean, std)      # used for the final model and the section 4 demo
X_test = apply_scaler(X_test_raw, mean, std)

## 4. Gradient computation (task parallelism)

**Linear regression in one paragraph.** The model predicts demand as `x . w + b`. Gradient descent starts at
`w = 0, b = 0`, measures how wrong the predictions are, and moves `w` and `b` a small step (the *learning rate*) in
the direction that reduces the error. It repeats this `GD_ITERATIONS` times. The "direction" is the **gradient**:
`sum((prediction - actual) * x)` over the rows. An L2 penalty gently pulls `w` toward 0.

**Why this parallelises.** The gradient is a *sum over rows*, and a sum can be split. We split the rows into
**6 zone shards** (`N_GRADIENT_SHARDS`): each shard holds *all the hours* for a group of zones. Every zone has the same number of
hours, so the shards are almost the same size (263 zones / 6 shards) and each worker gets about the same amount of work. Each iteration:

1. **Broadcast:** send the current `w` and `b` to every worker.
2. **Compute:** each worker sums `(prediction - actual) * x` over its own shard. Shards are independent: a worker reads only its own rows and the shared `w`, `b`.
3. **All-reduce:** add the shard sums together **in shard order** and divide by the total row count.
4. **Update:** take one step on the single shared model.

The shard sums add up to the same gradient as one pass over all rows, so the parallel model equals the sequential one
(up to tiny floating-point rounding, because the additions happen in a different order).
The shards are the same for 1, 2 or 3 workers, so every worker count gives *identical* weights.

Timings here are one run each. Person 3's section 6 repeats them for proper benchmarking.

In [ ]:
LR, L2, N_ITER = config.DEFAULT_LR, config.DEFAULT_L2, config.GD_ITERATIONS

t0 = time.perf_counter()
w_seq, b_seq = train_sequential(X_dev, y_dev, LR, L2, N_ITER)
seq_seconds = time.perf_counter() - t0

shards = make_shards(X_dev, y_dev, zones_dev)       # 6 zone shards
print("Shard sizes (rows):", [len(s_y) for _, s_y in shards])

rows = [{"workers": "sequential", "seconds": seq_seconds, "speedup": 1.0,
         "max |w - w_seq|": 0.0, "matches sequential": True}]
for n_workers in config.WORKER_COUNTS:
    t0 = time.perf_counter()
    w_par, b_par = train_parallel(shards, LR, L2, N_ITER, n_workers)
    seconds = time.perf_counter() - t0
    rows.append({"workers": n_workers, "seconds": seconds, "speedup": seq_seconds / seconds,
                 "max |w - w_seq|": float(np.abs(w_par - w_seq).max()),
                 "matches sequential": bool(np.allclose(w_par, w_seq) and np.isclose(b_par, b_seq))})

gradient_table = pd.DataFrame(rows)
print(gradient_table.to_string(index=False))
assert gradient_table["matches sequential"].all()

**Reading the table.** Every parallel model matches the sequential one (differences around 1e-15 are rounding).
Speedup is small or below 1: the whole development table is only about half a million rows, so one gradient pass is a
few milliseconds, while every iteration has to send `w`, `b` and the shard results between processes and wait for the slowest shard.
Those overheads (communication, synchronisation each iteration, process start-up) are what Person 3 analyses with Amdahl's law.

**Load imbalance here:** the shards are equal-sized, so each worker does about the same work. With 6 shards on 3 workers
each worker takes 2 shards per iteration; with 2 workers, 3 each; but a worker that is briefly slower still holds up
the others, because step 3 needs every shard before the next iteration can start.

## 5. Hyperparameter search (task parallelism)

A *hyperparameter* is a setting chosen before training (not learned from data). We try a 3 x 3 grid from `config.py`:
learning rate in {0.03, 0.1, 0.2} and L2 strength in {0, 0.001, 0.1}. That is **9 trials**.
(The learning rates stay below `2 / largest eigenvalue` of the scaled feature matrix, about 0.24, or gradient descent diverges.)

**Why the trials are independent:** each trial trains its own models from scratch on the same development rows and
reads only its own two settings. No trial needs another trial's result, so they can run on separate workers at the same
time, in any order.

**Why validation folds, not April:** the best setting is the one with the lowest **mean absolute error (MAE) averaged over the four
validation folds**, all inside January to March. If we chose on April, the final test score would be flattered by our own
choice. April is used once, at the end (section 8).

In [ ]:
search_data = {"X": X_dev_raw, "y": y_dev}     # raw features: each fold fits its own scaler on its own train rows

t0 = time.perf_counter()
seq_results = search_sequential(search_data, folds, N_ITER)
search_seq_seconds = time.perf_counter() - t0

search_rows = [{"workers": "sequential", "seconds": search_seq_seconds, "speedup": 1.0,
                "same 9 results": True}]
par_results = {}
for n_workers in config.WORKER_COUNTS:
    t0 = time.perf_counter()
    results = search_parallel(search_data, folds, N_ITER, n_workers)
    seconds = time.perf_counter() - t0
    par_results[n_workers] = results
    same = [(r["lr"], r["l2"], r["val_mae"]) for r in results] == \
           [(r["lr"], r["l2"], r["val_mae"]) for r in seq_results]
    search_rows.append({"workers": n_workers, "seconds": seconds,
                        "speedup": search_seq_seconds / seconds, "same 9 results": same})

search_table = pd.DataFrame(search_rows)
print(search_table.to_string(index=False))
assert search_table["same 9 results"].all()

In [ ]:
trial_table = pd.DataFrame(seq_results)[["lr", "l2", "val_mae", "seconds"]]
print(trial_table.to_string(index=False))

best = best_trial(seq_results)
print()
print(f"Best on validation folds: lr={best['lr']}, l2={best['l2']}, mean validation MAE={best['val_mae']:.3f} pickups")

imbalance = load_imbalance(par_results[config.MAX_WORKERS])
print(f"Trial times with {config.MAX_WORKERS} workers: slowest {imbalance['slowest']:.2f}s, "
      f"average {imbalance['mean']:.2f}s, slowest/average = {imbalance['ratio']:.2f}")

**Load imbalance between tasks.** If every trial took the same time, 9 trials on 3 workers would be 3 equal rounds.
`slowest/average` above shows how uneven they were. Every trial runs the same number of iterations on the same data, so the
times are close; with early stopping or different model sizes, some trials would finish early and leave workers idle
while one long trial finished. The split also matters: 9 trials on 3 workers is 3 full rounds, but on 2 workers it is 4.5 rounds,
so one worker idles during the last trial.

**Determinism:** each trial starts from `w = 0` with no random numbers and a fixed set of folds, and `pool.map` returns
results in grid order, so the table is identical every run. A tie would go to the earlier grid entry.

## 8. Final test results (April)

The best settings from section 5 train the final model on all development months (Jan to Mar), using the parallel gradient from
section 4. It is then scored **once** on April, which was never used for any decision.
The metrics are in pickups per zone-hour: **MAE** (average absolute error), **RMSE** (punishes big misses more) and
**WAPE** (total absolute error divided by total demand, a percentage that is easy to read).

Two simple baselines show whether the model is worth having: "same hour yesterday" (`lag_24`) and "same hour last week" (`lag_168`),
plus "always predict the training average".

In [ ]:
shards = make_shards(X_dev, y_dev, zones_dev)
w_final, b_final = train_parallel(shards, best["lr"], best["l2"], N_ITER, config.MAX_WORKERS)

final = scores(w_final, b_final, X_test, y_test)
lag24 = error_scores(test["lag_24"].to_numpy(), y_test)
lag168 = error_scores(test["lag_168"].to_numpy(), y_test)
average = error_scores(np.full(len(y_test), y_dev.mean()), y_test)

final_table = pd.DataFrame([
    {"model": "linear regression (parallel gradient)", **final},
    {"model": "baseline: same hour last week (lag_168)", **lag168},
    {"model": "baseline: same hour yesterday (lag_24)", **lag24},
    {"model": "baseline: training average", **average},
])
print(final_table.to_string(index=False))

**Is 300 steps enough?** Linear regression has an exact answer (least squares). It is not part of the model; it is only a check
that gradient descent has finished learning. If our result is about as good as the exact one, there is nothing more to gain from more iterations.

In [ ]:
A = np.c_[X_dev, np.ones(len(y_dev))]
solution = np.linalg.lstsq(A, y_dev, rcond=None)[0]
exact = scores(solution[:-1], solution[-1], X_test, y_test)
print(f"Test MAE: gradient descent {final['mae']:.3f}   exact least squares {exact['mae']:.3f}")

In [ ]:
# Numbers for the poster, written to a file so every poster number traces to this notebook.
from pathlib import Path
config.RESULTS_DIR.mkdir(exist_ok=True)
lines = [
    f"person2 target = pickups per zone per hour",
    f"person2 months: development {config.DEV_MONTHS}, test {config.TEST_MONTHS}",
    f"person2 rows development / test = {len(y_dev)} / {len(y_test)}",
    f"person2 features = {len(FEATURES)}: {', '.join(FEATURES)}",
    f"person2 validation folds = {config.VAL_FOLD_STARTS}, {config.VAL_FOLD_DAYS} days each",
    f"person2 gradient shards = {len(shards)} zone shards, iterations = {N_ITER}",
    f"person2 gradient sequential seconds = {seq_seconds:.3f}",
] + [f"person2 gradient {r['workers']} workers: {r['seconds']:.3f}s, speedup {r['speedup']:.2f}"
     for r in rows[1:]] + [
    f"person2 search sequential seconds = {search_seq_seconds:.3f}",
] + [f"person2 search {r['workers']} workers: {r['seconds']:.3f}s, speedup {r['speedup']:.2f}"
     for r in search_rows[1:]] + [
    f"person2 best lr = {best['lr']}, l2 = {best['l2']}, mean validation MAE = {best['val_mae']:.3f}",
    f"person2 trial time slowest/average = {imbalance['ratio']:.2f}",
    f"person2 test MAE = {final['mae']:.3f}, RMSE = {final['rmse']:.3f}, WAPE = {final['wape']:.1%}",
    f"person2 baseline lag_168 test MAE = {lag168['mae']:.3f}, RMSE = {lag168['rmse']:.3f}, WAPE = {lag168['wape']:.1%}",
    f"person2 baseline lag_24 test MAE = {lag24['mae']:.3f}, RMSE = {lag24['rmse']:.3f}, WAPE = {lag24['wape']:.1%}",
    f"person2 baseline average test MAE = {average['mae']:.3f}, RMSE = {average['rmse']:.3f}, WAPE = {average['wape']:.1%}",
    f"person2 exact least squares test MAE = {exact['mae']:.3f}",
]
(config.RESULTS_DIR / "person2_numbers.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))